In [1]:
from document_extractor import extract_itac_report

doc_1_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report1/LS2502 - Final Draft R2.docx"
doc_2_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report2/LS2508 - Final Draft.docx"
out = extract_itac_report(doc_2_path, output="html", save_files=True)



In [2]:
out.keys()

dict_keys(['general_information', 'annual_energy_usages_and_costs', 'carbon_footprint', 'recommendation_summary_table', 'ar_summary', 'assessment_recommendations'])

In [3]:
recommendations = out['assessment_recommendations']

In [4]:
# Test the new get_single_ar_summary_table function
from doc_extractor_utils import get_single_ar_summary_table

# Extract data from the first AR
ar_1_data = get_single_ar_summary_table(recommendations[0])
ar_1_data


{'ar_number': '1',
 'headers': ['Energy Savings (kWh/yr)',
  'Energy Cost Savings($/yr)',
  'Demand Savings (kW/yr)',
  'Demand Cost($/yr)',
  'Total Cost Savings ($/yr)',
  'CO2Reduction (tons/yr)',
  'Imp. Cost ($)',
  'Payback Period(yrs)'],
 'standardized_headers': ['electricity_savings_kwh_per_year',
  'energy_cost_savingsdollar_per_yr',
  'demand_savings_kw_per_year',
  'demand_costdollar_per_yr',
  'total_cost_savings_per_year',
  'co2reduction_tons_per_yr',
  'implementation_cost',
  'payback_period_years'],
 'data': {'electricity_savings_kwh_per_year': 6806,
  'energy_cost_savingsdollar_per_yr': 694,
  'demand_savings_kw_per_year': 17,
  'demand_costdollar_per_yr': 77,
  'total_cost_savings_per_year': 771,
  'co2reduction_tons_per_yr': 3,
  'implementation_cost': 675,
  'payback_period_years': 0.88}}

In [5]:
# Now let's get the recommendation summary table and compare
from doc_extractor_utils import get_recommended_summary_table_json

rec_summary = get_recommended_summary_table_json(out['recommendation_summary_table'])

# Get the first recommendation from the summary table
first_rec_from_summary = rec_summary['recommendations'][0]

print("From individual AR Savings Summary:")
print(ar_1_data['data'])
print("\nFrom Recommendation Summary Table:")
print(first_rec_from_summary)


From individual AR Savings Summary:
{'electricity_savings_kwh_per_year': 6806, 'energy_cost_savingsdollar_per_yr': 694, 'demand_savings_kw_per_year': 17, 'demand_costdollar_per_yr': 77, 'total_cost_savings_per_year': 771, 'co2reduction_tons_per_yr': 3, 'implementation_cost': 675, 'payback_period_years': 0.88}

From Recommendation Summary Table:
{'ar_number': '1', 'category': 'Lighting', 'description': 'Utilize Higher Efficiency Lamps and/or Ballasts', 'electricity_savings_kwh_per_year': 6806, 'energy_cost_savings_per_year': 694, 'demand_savings_kw_per_year': 17, 'demandcost_savings_dollar_per_yr': 77, 'admin_cost_savingsdollar_per_yr': 0, 'propane_savingsmmbtu_per_yr': 0, 'propanecost_savingdollar_per_yr': 0, 'total_cost_savings_per_year': 771, 'co2_reduction_tons_per_year': 3, 'implementation_cost': 675, 'payback_period_years': 0.88}


In [9]:
# Function to compare AR data with recommendation summary table
def compare_ar_with_summary(ar_data, summary_rec):
    """Compare data from individual AR with the corresponding row in summary table."""
    
    differences = []
    matches = []
    
    # Get the data dict from AR
    ar_values = ar_data['data']
    
    # Compare common fields
    for field in ar_values.keys():
        if field in summary_rec:
            ar_val = ar_values[field]
            summary_val = summary_rec[field]
            
            # Compare values with tolerance for floats
            if isinstance(ar_val, (int, float)) and isinstance(summary_val, (int, float)):
                if abs(ar_val - summary_val) < 0.01:
                    matches.append({
                        'field': field,
                        'ar_value': ar_val,
                        'summary_value': summary_val,
                        'match': True
                    })
                else:
                    differences.append({
                        'field': field,
                        'ar_value': ar_val,
                        'summary_value': summary_val,
                        'difference': abs(ar_val - summary_val)
                    })
            elif ar_val == summary_val:
                matches.append({
                    'field': field,
                    'ar_value': ar_val,
                    'summary_value': summary_val,
                    'match': True
                })
            else:
                differences.append({
                    'field': field,
                    'ar_value': ar_val,
                    'summary_value': summary_val,
                    'difference': 'type mismatch or different values'
                })
    
    return {
        'matches': matches,
        'differences': differences,
        'total_matches': len(matches),
        'total_differences': len(differences)
    }

# Compare AR 1
comparison = compare_ar_with_summary(ar_1_data, rec_summary['recommendations'][0])

print(f"Total matches: {comparison['total_matches']}")
print(f"Total differences: {comparison['total_differences']}")

if comparison['differences']:
    print("\nDifferences found:")
    for diff in comparison['differences']:
        print(f"  {diff['field']}: AR={diff['ar_value']} vs Summary={diff['summary_value']}")
else:
    print("\nAll values match!")


Total matches: 5
Total differences: 0

All values match!


In [7]:
from IPython.display import HTML
HTML(recommendations[0])




